# HGNN-SA: Baseline Reproduction (Experiment A)

## Scientific Objective
Faithfully reproduce the original HODDI paper's **HGNN-SA baseline** across the paper's **4 random seeds** (`[42, 3407, 54321, 123456]`).

### Target Benchmark Values (HODDI Paper Table 7):
- **ROC-AUC:** $0.957 \pm 0.003$
- **PR-AUC:** $0.939 \pm 0.008$
- **F1-Score:** $0.933 \pm 0.001$
- **Precision:** $0.906 \pm 0.002$

### Key Reproducibility Criteria:
1. **Exact Drug Universe:** Strict intersection with the official DrugBank SMILES dictionary yielding the exact $10,250$ drugs used by the authors.
2. **Authentic Hypergraph Architecture:** 1-layer `HypergraphConv` with 3 attention heads, learned hyperedge attributes from incidence pattern projections, and ChemBERTa 768d representations.
3. **Author Training Protocol:** AdamW ($lr=0.0005, wd=0.001$), CrossEntropyLoss (2-class), batch size 64, 100 epochs, model checkpointing strictly on validation F1 score.

In [ ]:
import os
import ast
import copy
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as hnn
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Compute device: {device}")

## 1. HGNN-SA Model Architecture
Matches `demo4_HGNN/CLOSEgaps.py` with `enable_hygnn=False`, $L=1$ hypergraph convolution.

In [ ]:
class HGNN_SA(nn.Module):
    def __init__(self, input_num, input_feature_num, emb_dim=128, conv_dim=64, head=3, p=0.1):
        super(HGNN_SA, self).__init__()
        self.emb_dim = emb_dim
        self.conv_dim = conv_dim
        self.head = head
        self.p = p
        
        self.linear_encoder = nn.Linear(input_feature_num, emb_dim)
        self.pre_linear = nn.Linear(768, emb_dim) 
        self.in_channel = 2 * emb_dim
        
        self.relu = nn.ReLU()
        self.hypergraph_conv = hnn.HypergraphConv(
            self.in_channel, conv_dim, heads=head, use_attention=True, dropout=p
        )
        self.hyper_attr_liner = nn.Linear(input_num, self.in_channel)
        self.hyperedge_linear = nn.Linear(conv_dim * head, 2)
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, input_features, incidence_matrix, extra_feature):
        incidence_matrix_T = incidence_matrix.T
        
        # 1. Structural incidence encoding
        input_nodes_features = self.relu(self.linear_encoder(input_features))
        
        # 2. Extract edge connectivity for PyG HypergraphConv
        row, col = torch.where(incidence_matrix_T)
        edges = torch.cat((col.view(1, -1), row.view(1, -1)), dim=0).to(incidence_matrix.device)
        
        # 3. Learned hyperedge attributes from incidence pattern
        hyperedge_attr = self.hyper_attr_liner(incidence_matrix_T)
        
        # 4. Chemical feature encoding and concatenation
        extra_feat = self.relu(self.pre_linear(extra_feature))
        input_nodes_features = torch.cat((extra_feat, input_nodes_features), dim=1)
        
        # 5. Hypergraph convolution message passing
        input_nodes_features = self.hypergraph_conv(
            input_nodes_features, edges, hyperedge_attr=hyperedge_attr
        )
        
        # 6. Readout: sum pooling over hyperedges
        hyperedge_feature = torch.mm(incidence_matrix_T, input_nodes_features)
        
        # 7. Classification logits (2-class)
        return self.hyperedge_linear(hyperedge_feature)

    def predict(self, input_features, incidence_matrix, extra_feature):
        return self.softmax(self.forward(input_features, incidence_matrix, extra_feature))

## 2. Dataset Loading & Drug Indexing
Faithfully reads the $29$ train quarters, $6$ validation quarters, and $6$ test quarters.

In [ ]:
def find_file(filename, search_roots=['/kaggle/input', '.', '..', '../..', 'dataset']):
    for root_dir in search_roots:
        if os.path.exists(root_dir):
            for root, dirs, files in os.walk(root_dir):
                if filename in files:
                    return os.path.join(root, filename)
    return None

def find_dir(dirname, search_roots=['/kaggle/input', '.', '..', '../..', 'dataset']):
    for root_dir in search_roots:
        if os.path.exists(root_dir):
            for root, dirs, files in os.walk(root_dir):
                if dirname in dirs:
                    return os.path.join(root, dirname)
    return None

def find_dataset_path():
    p = find_file('drug_embeddings_768d.pt')
    if p:
        return os.path.dirname(p)
    d = find_dir('subset_drug2-8_SE5-50')
    if d:
        return os.path.dirname(os.path.dirname(d))
    return 'dataset'

def load_data(base_path, device):
    subset_dir = find_dir('subset_drug2-8_SE5-50')
    if subset_dir is None:
        raise FileNotFoundError("Could not locate 'subset_drug2-8_SE5-50' directory across search paths.")
    print(f"Using evaluation subset directory: {subset_dir}")

    # 1. SMILES dictionary
    dict_path = find_file('Drugbank_ID_SMILE_all_structure links.csv')
    if dict_path is None:
        raise FileNotFoundError("Could not find 'Drugbank_ID_SMILE_all_structure links.csv'")
    print(f"Loading SMILES mapping from: {dict_path}")
    smiles_ds = pd.read_csv(dict_path)
    drugbank_to_smiles = smiles_ds.dropna(subset=['DrugBank ID', 'SMILES']).set_index('DrugBank ID')['SMILES'].to_dict()

    # 2. Merged dataset to identify unique drug set
    merged_dir = os.path.join(subset_dir, 'merged_subset')
    pos_step6 = os.path.join(merged_dir, 'positive_samples_2014Q3_2024Q3_step6.csv')
    neg_step6 = os.path.join(merged_dir, 'negative_samples_2014Q3_2024Q3_step6.csv')
    all_ds = pd.concat([pd.read_csv(pos_step6), pd.read_csv(neg_step6)], axis=0)

    all_drugs = set()
    for drug_ids in all_ds['DrugBankID']:
        all_drugs.update([d for d in ast.literal_eval(drug_ids) if d.lower() != 'none'])

    drug_to_index_raw = {drug: idx for idx, drug in enumerate(all_drugs)}
    raw_drug_id_list = [None] * len(all_drugs)
    for did in drug_to_index_raw:
        raw_drug_id_list[drug_to_index_raw[did]] = did

    # Exact 10,250 drug universe matching HODDI paper
    drug_id_list = [did for did in raw_drug_id_list if did in drugbank_to_smiles]
    drug_to_index = {did: idx for idx, did in enumerate(drug_id_list)}
    num_drugs = len(drug_id_list)
    print(f"Total unique drugs after SMILES filtering: {num_drugs}")

    # 3. Quarterly splits
    training_sub_ds_names = [
        '2015Q1', '2015Q2', '2015Q3', '2016Q4', '2017Q1', '2017Q2', '2017Q3', '2017Q4',
        '2018Q3', '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3',
        '2020Q4', '2021Q1', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4',
        '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q2'
    ]
    validating_sub_ds_names = ['2014Q3', '2015Q4', '2016Q1', '2016Q3', '2021Q2', '2024Q1']
    testing_sub_ds_names = ['2014Q4', '2016Q2', '2018Q1', '2018Q2', '2018Q4', '2024Q3']

    def merge_quarters(sub_datasets):
        pos_merged, neg_merged = [], []
        for sub_ds in sub_datasets:
            pos_f = os.path.join(subset_dir, f'{sub_ds}_positive_samples_condition123_SE_above_0.9.csv')
            neg_f = os.path.join(subset_dir, f'{sub_ds}_negative_samples_condition123_SE_above_0.9.csv')
            if os.path.exists(pos_f):
                pos_merged.append(pd.read_csv(pos_f))
            if os.path.exists(neg_f):
                neg_merged.append(pd.read_csv(neg_f))
        return pd.concat(pos_merged, axis=0), pd.concat(neg_merged, axis=0)

    def build_incidence_pair(pos_df, neg_df):
        num_pos, num_neg = len(pos_df), len(neg_df)
        inc_pos = np.zeros((num_drugs, num_pos), dtype=np.float32)
        inc_neg = np.zeros((num_drugs, num_neg), dtype=np.float32)
        for col_idx, drug_list in enumerate(pos_df['DrugBankID']):
            for did in ast.literal_eval(drug_list):
                if did in drug_to_index:
                    inc_pos[drug_to_index[did], col_idx] = 1.0
        for col_idx, drug_list in enumerate(neg_df['DrugBankID']):
            for did in ast.literal_eval(drug_list):
                if did in drug_to_index:
                    inc_neg[drug_to_index[did], col_idx] = 1.0
        return inc_pos, inc_neg, np.ones(num_pos, dtype=np.int64), np.zeros(num_neg, dtype=np.int64)

    print("Merging quarterly CSV files...")
    train_pos, train_neg = merge_quarters(training_sub_ds_names)
    val_pos, val_neg = merge_quarters(validating_sub_ds_names)
    test_pos, test_neg = merge_quarters(testing_sub_ds_names)

    train_inc_pos, train_inc_neg, train_lpos, train_lneg = build_incidence_pair(train_pos, train_neg)
    val_inc_pos, val_inc_neg, val_lpos, val_lneg = build_incidence_pair(val_pos, val_neg)
    test_inc_pos, test_inc_neg, test_lpos, test_lneg = build_incidence_pair(test_pos, test_neg)

    # 4. Load ChemBERTa embeddings
    emb_path = find_file('drug_embeddings_768d.pt')
    if emb_path is None:
        raise FileNotFoundError("Could not find 'drug_embeddings_768d.pt'")
    print(f"Loading embeddings from: {emb_path}")
    cached_emb = torch.load(emb_path, map_location='cpu', weights_only=True)
    
    # Compute missing if needed
    missing = [d for d in drug_id_list if d not in cached_emb and d in drugbank_to_smiles]
    if len(missing) > 0:
        print(f"Computing embeddings for {len(missing)} missing drugs...")
        from transformers import AutoModelForMaskedLM, AutoTokenizer
        m_name = 'seyonec/PubChem10M_SMILES_BPE_450k'
        local_dir = find_dir('PubChem10M_SMILES_BPE_450k')
        if local_dir:
            m_name = local_dir
        tok = AutoTokenizer.from_pretrained(m_name)
        m = AutoModelForMaskedLM.from_pretrained(m_name).to(device)
        m.eval()
        with torch.no_grad():
            for i in range(0, len(missing), 64):
                b_ids = missing[i:i+64]
                b_smiles = [drugbank_to_smiles[x] for x in b_ids]
                inp = tok(b_smiles, return_tensors='pt', max_length=256, padding='max_length', truncation=True).to(device)
                vecs = m(**inp, output_hidden_states=True).hidden_states[-1].mean(dim=1).cpu()
                for x, v in zip(b_ids, vecs):
                    cached_emb[x] = v

    extra_feature = torch.zeros((num_drugs, 768), dtype=torch.float32)
    for idx, did in enumerate(drug_id_list):
        if did in cached_emb:
            extra_feature[idx] = cached_emb[did]

    return {
        'train_inc_pos': torch.tensor(train_inc_pos, dtype=torch.float32).to(device),
        'train_inc_all': torch.tensor(np.concatenate([train_inc_pos, train_inc_neg], axis=1), dtype=torch.float32),
        'y_train_all': np.concatenate([train_lpos, train_lneg]),
        'val_inc_all': torch.tensor(np.concatenate([val_inc_pos, val_inc_neg], axis=1), dtype=torch.float32).to(device),
        'y_val_all': np.concatenate([val_lpos, val_lneg]),
        'test_inc_all': torch.tensor(np.concatenate([test_inc_pos, test_inc_neg], axis=1), dtype=torch.float32).to(device),
        'y_test_all': np.concatenate([test_lpos, test_lneg]),
        'extra_feature': extra_feature.to(device),
        'num_drugs': num_drugs,
    }

base_path = find_dataset_path()
data = load_data(base_path, device)

## 3. Multi-Seed Training Loop (`[42, 3407, 54321, 123456]`)

In [ ]:
def train_and_eval_seed(seed, data, device, epochs=100, batch_size=64, lr=0.0005, weight_decay=0.001):
    set_random_seed(seed)
    print(f"\n>>> Training Seed {seed}...")

    train_inc_all = data['train_inc_all']
    y_train_all = data['y_train_all']
    num_train_samples = train_inc_all.shape[1]

    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(num_train_samples, generator=g)
    train_inc_shuffled = train_inc_all[:, perm]
    y_train_shuffled = torch.tensor(y_train_all[perm.numpy()], dtype=torch.long)

    model = HGNN_SA(
        input_num=data['num_drugs'],
        input_feature_num=data['train_inc_pos'].shape[1],
        emb_dim=128, conv_dim=64, head=3, p=0.1
    ).to(device)
    model.apply(init_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_val_f1 = 0.0
    best_weights = None
    num_batches = num_train_samples // batch_size

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for b in range(num_batches):
            optimizer.zero_grad()
            batch_inc = train_inc_shuffled[:, b*batch_size:(b+1)*batch_size].to(device)
            batch_y = y_train_shuffled[b*batch_size:(b+1)*batch_size].to(device)
            y_pred = model(data['train_inc_pos'], batch_inc, data['extra_feature'])
            loss = criterion(y_pred, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_probs = model.predict(data['train_inc_pos'], data['val_inc_all'], data['extra_feature'])
            val_scores = val_probs[:, 1].cpu().numpy()
            
        b_score = (val_scores >= 0.5).astype(int)
        val_f1 = f1_score(data['y_val_all'], b_score, zero_division=0)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())

        if epoch % 20 == 0 or epoch == epochs:
            print(f"  Epoch {epoch:03d} | Loss: {epoch_loss/num_batches:.4f} | Val F1: {val_f1:.4f} (Best: {best_val_f1:.4f})")

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        test_probs = model.predict(data['train_inc_pos'], data['test_inc_all'], data['extra_feature'])
        test_scores = test_probs[:, 1].cpu().numpy()

    y_test = data['y_test_all']
    b_preds = (test_scores >= 0.5).astype(int)

    res = {
        'seed': seed,
        'precision': precision_score(y_test, b_preds, zero_division=0),
        'recall': recall_score(y_test, b_preds, zero_division=0),
        'f1': f1_score(y_test, b_preds, zero_division=0),
        'auc': roc_auc_score(y_test, test_scores),
        'prauc': average_precision_score(y_test, test_scores),
        'test_scores': test_scores,
    }
    print(f"Seed {seed} -> AUC: {res['auc']:.4f}, PR-AUC: {res['prauc']:.4f}, F1: {res['f1']:.4f}")
    return res

seeds = [42, 3407, 54321, 123456]
all_seed_results = []
for s in seeds:
    r = train_and_eval_seed(s, data, device)
    all_seed_results.append(r)

## 4. Benchmark Summary & Verification against HODDI Paper Table 7

In [ ]:
res_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'test_scores'} for r in all_seed_results])
res_df.to_csv('exp_a_hgnn_sa_reproduction_results.csv', index=False)

paper_table_7 = {
    'precision': '0.906 ± 0.002',
    'f1': '0.933 ± 0.001',
    'auc': '0.957 ± 0.003',
    'prauc': '0.939 ± 0.008',
    'recall': 'N/A'
}

summary_rows = []
for metric in ['precision', 'recall', 'f1', 'auc', 'prauc']:
    m_mean = res_df[metric].mean()
    m_std = res_df[metric].std()
    summary_rows.append({
        'Metric': metric.upper(),
        'Our Reproduction (4 Seeds)': f"{m_mean:.4f} ± {m_std:.4f}",
        'HODDI Paper (Table 7)': paper_table_7.get(metric, 'N/A')
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_markdown(index=False))
display(summary_df)